In [37]:
import h5py
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from transformers import AutoModel
from utils.CustomDataset import BaselineDataset
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

In [2]:
# Load Pretrained DINOv2 Model
dino_model = AutoModel.from_pretrained("facebook/dinov2-small")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dino_model.to(device)
# LoRA Configuration
lora_config = LoraConfig(
    r=8,  # Rank
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]  # Target attention layers
)

# Apply LoRA to DINOv2 Model
dino_lora = get_peft_model(dino_model, lora_config)

# Define Binary Classification Head
class BinaryClassifier(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model
        self.fc = nn.Linear(384, 1)  # 384-dim for DINOv2-small

    def forward(self, x):
        x = self.base_model(x).last_hidden_state[:, 0, :]
        x = self.fc(x)
        return x

# Wrap Model
model = BinaryClassifier(dino_lora).to(device)


In [ ]:

# Dataset & DataLoader
def get_dataloader(batch_size=16):
    transform = transforms.Compose([
        transforms.Resize((98, 98))
    ])
    dataset = BaselineDataset(
        "../data_test/train.h5",
        preprocessing=transform, mode='train'
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

dataloader = get_dataloader()

# Loss & Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-5)

# Training Loop
for epoch in range(3):  # Train for 3 epochs
    model.train()
    for images, labels in tqdm(dataloader):
        images, labels = images.to(device), labels.to(device, dtype=torch.float32)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

torch.save(model.state_dict(), "dino_lora_finetuned.pth")
print("Fine-tuning complete! Model saved as dino_lora_finetuned.pth")


  0%|          | 20/6250 [00:25<2:14:20,  1.29s/it]


KeyboardInterrupt: 

In [ ]:

model.load_state_dict(torch.load("dino_lora_finetuned.pth", map_location=torch.device('cpu')))
transform = transforms.Compose([
        transforms.Resize((98, 98))
    ])
val_dataset = BaselineDataset(
    "../data_test/val.h5",
    preprocessing=transform, mode='train'
)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)
model.eval()

BinaryClassifier(
  (base_model): PeftModel(
    (base_model): LoraModel(
      (model): Dinov2Model(
        (embeddings): Dinov2Embeddings(
          (patch_embeddings): Dinov2PatchEmbeddings(
            (projection): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
          )
          (dropout): Dropout(p=0.0, inplace=False)
        )
        (encoder): Dinov2Encoder(
          (layer): ModuleList(
            (0-11): 12 x Dinov2Layer(
              (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
              (attention): Dinov2Attention(
                (attention): Dinov2SelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=384, out_features=384, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=384, out_features=8, bia

In [5]:
correct = 0
total = 0

for images, labels in tqdm(val_dataloader):
    images, labels = images.to(device), labels.to(device, dtype=torch.float32)
    with torch.no_grad():
        outputs = model(images).squeeze()
        predicted = torch.sigmoid(outputs) > 0.5
        correct += (predicted == labels.byte()).sum().item()
        total += labels.size(0)
accuracy = correct / total

  0%|          | 0/2182 [00:00<?, ?it/s]

In [ ]:
transform = transforms.Compose([
        transforms.Resize((98, 98))
    ])
test_dataset = BaselineDataset(
    "../data_test/test.h5",
    preprocessing=transform, mode='val'
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
model.eval()

In [ ]:
TEST_IMAGES_PATH = '../data_test/test.h5'
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    test_ids = list(hdf.keys())

In [ ]:
TEST_IMAGES_PATH = '../data_test/test.h5'
solutions_data = {'ID': [], 'Pred': []}
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    for test_id in tqdm(test_ids):
        img = transform(torch.tensor(np.array(hdf.get(test_id).get('img'))).unsqueeze(0).float())
        pred = model(img.to(device)).detach().cpu()
        solutions_data['ID'].append(int(test_id))
        solutions_data['Pred'].append(int(pred.item() > 0.5))


  0%|          | 0/85054 [00:00<?, ?it/s]

NameError: name 'pd' is not defined

In [ ]:

solutions_data = pd.DataFrame(solutions_data).set_index('ID')
solutions_data.to_csv('fine_tuned_dino.csv')

## Adversarial

In [25]:
from transformers import AutoModel
dino_model = AutoModel.from_pretrained("facebook/dinov2-small")

dino_model.to(device)
# LoRA Configuration
lora_config = LoraConfig(
    r=4,  # Rank
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"]  # Target attention layers
)

# Apply LoRA to DINOv2 Model
dino_lora = get_peft_model(dino_model, lora_config)

In [154]:
class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x) # Avoid modifying the input tensor directly

    @staticmethod
    def backward(ctx, grad_output):
        # Return the negative gradient multiplied by alpha, and None for alpha's gradient
        return grad_output.neg() * ctx.alpha, None
        
class AdversarialClassifier(nn.Module):
    def __init__(self, feature_extractor, embedding_dim, num_classes, num_centers, lambda_=0.1):
        super().__init__()
        self.fc = torch.nn.Sequential(
            torch.nn.Linear(embedding_dim, 256),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU(),
            torch.nn.Linear(256, 128),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1)
        ) # Classification head
        self.grl = GradientReversalLayer
        self.center_fc = torch.nn.Sequential(
            torch.nn.Linear(embedding_dim, 128),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 64),
            torch.nn.Dropout(0.5),
            torch.nn.ReLU(),
            torch.nn.Linear(64, num_centers))
        self.feature_extractor = feature_extractor
        self.lambda_ = lambda_
    
    def forward(self, x):
        features = self.feature_extractor(x).last_hidden_state[:, 0, :]
        class_pred = self.fc(features)
        rev_x = self.grl.apply(features, self.lambda_)  # Reverse gradients for adversarial learning
        center_pred = self.center_fc(rev_x)
        return class_pred, center_pred
    
# Wrap Model
model = AdversarialClassifier(dino_lora, 384, 1, 3).to(device)

In [138]:

model.load_state_dict(torch.load("dino_lora_adversarial.pth", map_location=torch.device('cpu')))
transform = transforms.Compose([
        transforms.Resize((98, 98))
    ])
val_dataset = BaselineDataset(
    "../data_test/val.h5",
    preprocessing=transform, mode='train'
)
val_dataloader = DataLoader(val_dataset, batch_size=16, shuffle=False)
model.eval()

AdversarialClassifier(
  (fc): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): Dropout(p=0.5, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): Dropout(p=0.5, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): Dropout(p=0.5, inplace=False)
    (8): ReLU()
    (9): Linear(in_features=64, out_features=1, bias=True)
  )
  (center_fc): Sequential(
    (0): Linear(in_features=384, out_features=128, bias=True)
    (1): Dropout(p=0.5, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Dropout(p=0.5, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=3, bias=True)
  )
  (feature_extractor): PeftModel(
    (base_model): LoraModel(
      (model): Dinov2Model(
        (embeddings): Dinov2Embeddings(
          (patch_embeddings): Dinov2PatchEmbeddings(
            (projection): Conv2d(3, 3

In [146]:
for i in range(len(val_dataset)):
    img, label = val_dataset[i]
    if label == [0]:
        print(i)

8283
8284
8285
8286
8287
8288
8289
8290
8292
8293
8294
8295
8296
8297
8298
8299
8300
8301
8303
8304
8305
8306
8307
8308
8309
8310
8311
8312
8314
8315
8316
8317
8318
8319
8320
8321
8322
8323
8325
8326
8327
8328
8329
8330
8331
8332
8333
8334
8337
8338
8339
8340
8341
8342
8343
8344
8345
8346
8348
8349
8350
8351
8352
8353
8354
8355
8356
8357
8359
8360
8361
8362
8363
8364
8365
8366
8367
8368
8370
8371
8372
8373
8374
8375
8376
8377
8378
8379
8381
8382
8383
8384
8385
8386
8387
8388
8389
8390
8392
8393
8394
8395
8396
8397
8398
8399
8400
8401
8403
8404
8405
8406
8407
8408
8409
8410
8411
8412
8414
8415
8416
8417
8418
8419
8420
8421
8422
8423
8425
8426
8427
8428
8429
8430
8431
8432
8433
8434
8436
8437
8438
8439
8440
8441
8442
8443
8444
8445
8448
8449
8450
8451
8452
8453
8454
8455
8456
8457
8459
8460
8461
8462
8463
8464
8465
8466
8467
8468
8470
8471
8472
8473
8474
8475
8476
8477
8478
8479
8481
8482
8483
8484
8485
8486
8487
8488
8489
8490
8492
8493
8494
8495
8496
8497
8498
8499
8500
8501
8503
8504


KeyboardInterrupt: 

In [149]:
model(val_dataset[8309][0].unsqueeze(0).to(device))

(tensor([[48302220.]], grad_fn=<AddmmBackward0>),
 tensor([[23.6436, -5.8804, -6.0264]], grad_fn=<AddmmBackward0>))

In [150]:
correct = 0
total = 0

for images, labels in tqdm(val_dataloader):
    images, labels = images.to(device), labels.to(device, dtype=torch.float32)
    with torch.no_grad():
        predicted = model.fc(model.feature_extractor(images.to(device)).last_hidden_state[:, 0, :]).detach().cpu()
        predicted = torch.nn.Sigmoid()(predicted) > 0.5
        correct += (predicted.flatten() == labels.byte()).sum().item()
        total += labels.size(0)
        if total % 10000 == 0:
            print(f"Total: {total}, Correct: {correct}, Accuracy: {correct / total:.4f}")
accuracy = correct / total

  0%|          | 0/2182 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [156]:
model.load_state_dict(torch.load("dino_lora_adversarial_3epochs_lambda1.pth", map_location=torch.device('cpu')))
transform = transforms.Compose([
        transforms.Resize((98, 98))
    ])
test_dataset = BaselineDataset(
    "../data_test/test.h5",
    preprocessing=transform, mode='val'
)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)
model.eval()

AdversarialClassifier(
  (fc): Sequential(
    (0): Linear(in_features=384, out_features=256, bias=True)
    (1): Dropout(p=0.5, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): Dropout(p=0.5, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=128, out_features=64, bias=True)
    (7): Dropout(p=0.5, inplace=False)
    (8): ReLU()
    (9): Linear(in_features=64, out_features=1, bias=True)
  )
  (center_fc): Sequential(
    (0): Linear(in_features=384, out_features=128, bias=True)
    (1): Dropout(p=0.5, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): Dropout(p=0.5, inplace=False)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=3, bias=True)
  )
  (feature_extractor): PeftModel(
    (base_model): LoraModel(
      (model): Dinov2Model(
        (embeddings): Dinov2Embeddings(
          (patch_embeddings): Dinov2PatchEmbeddings(
            (projection): Conv2d(3, 3

In [157]:
TEST_IMAGES_PATH = '../data_test/test.h5'
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    test_ids = list(hdf.keys())

In [158]:
TEST_IMAGES_PATH = '../data_test/test.h5'
solutions_data = {'ID': [], 'Pred': []}
with h5py.File(TEST_IMAGES_PATH, 'r') as hdf:
    for test_id in tqdm(test_ids):
        img = transform(torch.tensor(np.array(hdf.get(test_id).get('img'))).unsqueeze(0).float())
        pred = model.fc(model.feature_extractor(img.to(device)).last_hidden_state[:, 0, :]).detach().cpu()
        solutions_data['ID'].append(int(test_id))
        solutions_data['Pred'].append(int(pred.item() > 0.5))
solutions_data = pd.DataFrame(solutions_data).set_index('ID')
solutions_data.to_csv('fine_tuned_dino_lambda1.csv') 

  0%|          | 0/85054 [00:00<?, ?it/s]